# W03 — Data Contract

Lane: `writing-data-contracts`  
Dataset: `flyrank/flyrank-data`  
Main table: `fact_content_daily_performance`

Goal:
- Define the contract in plain words.
- Verify the grain, slice size, and availability.
- Build a small honest feature frame.
- Demonstrate a leakage trap, then remove it.
- State one limitation.


In [ ]:
import pandas as pd
from IPython.display import display, Markdown


## 1) Data contract

1. One row means: one content item for one client on one report date.
2. Tables I will use: `fact_content_daily_performance` and, if needed, static lookup tables such as `dim_content` and `dim_clients`.
3. Time window: one mid-panel month, for example `2026-03`.
4. What I will predict or rank: a safe future performance proxy for each content item, using only information available before the decision moment.
5. What I exclude: future-looking columns, label-derived columns, and anything that already contains the answer.


In [ ]:
month = '2026-03'


## 2) Verification query 1 — grain


In [ ]:
-- Grain check: should return no rows if one row = one client/content/date record
SELECT
  report_date,
  client_id,
  content_id,
  COUNT(*) AS rows_per_key
FROM `fact_content_daily_performance`
WHERE month = '2026-03'
GROUP BY 1, 2, 3
HAVING COUNT(*) > 1
ORDER BY rows_per_key DESC
LIMIT 20;


If this returns no rows, the grain matches the contract. If it returns rows, the table has duplicates at the key I claimed.


## 2) Verification query 2 — row count and date span


In [ ]:
SELECT
  COUNT(*) AS row_count,
  MIN(report_date) AS min_date,
  MAX(report_date) AS max_date
FROM `fact_content_daily_performance`
WHERE month = '2026-03';


This proves how many rows are in the chosen month and the exact date span covered by the slice.


## 2) Verification query 3 — availability


In [ ]:
SELECT
  COUNT(*) AS total_rows,
  COUNTIF(is_available IS TRUE) AS available_rows
FROM `fact_content_daily_performance`
WHERE month = '2026-03';


This shows how many rows survive the availability filter using `IS TRUE`.


## 3) Five features

1. `content_age_days`
   - Knowable at the decision moment because it depends only on publish date and current report date.

2. `prev_7d_impressions`
   - Knowable at the decision moment because it uses only past observations.

3. `prev_7d_clicks`
   - Knowable at the decision moment because it uses only past observations.

4. `click_through_rate_prev7d`
   - Knowable at the decision moment because it is computed from already observed clicks and impressions.

5. `client_content_share`
   - Knowable at the decision moment because it uses only historical client-level content distribution.


In [ ]:
-- Replace column names if your warehouse uses different names.
WITH base AS (
  SELECT *
  FROM `fact_content_daily_performance`
  WHERE month = '2026-03'
    AND is_available IS TRUE
),
features AS (
  SELECT
    report_date,
    client_id,
    content_id,
    DATE_DIFF(report_date, publish_date, DAY) AS content_age_days,
    prev_7d_impressions,
    prev_7d_clicks,
    SAFE_DIVIDE(prev_7d_clicks, prev_7d_impressions) AS click_through_rate_prev7d,
    client_content_share,
    target
  FROM base
)
SELECT *
FROM features
LIMIT 10;


## 4) Leakage trap

First I add a label-derived column on purpose, then I check the score. The score jumps unrealistically because the model can see the answer. Then I remove the leaked column and keep the honest score.


In [ ]:
# Example template only.
# Replace df with your actual feature frame once you run the SQL above.

# df['leak'] = df['target']

# quick_score_with_leak = ...
# quick_score_without_leak = ...

# display(Markdown(f'Score with leak: {quick_score_with_leak}'))
# display(Markdown(f'Score without leak: {quick_score_without_leak}'))


## 4) Limitation

This notebook uses only one month, so it may miss seasonality, long-term behavior, and rare patterns. That means the slice is useful for contract writing and feature design, but not enough to claim full model generalization.


## 5) Self-check

- I defined the row grain in plain words.
- I listed the tables I will use.
- I chose one mid-panel month.
- I named the prediction or ranking target.
- I named one deliberate exclusion.
- I ran exactly three verification queries.
- I used `IS TRUE` for availability.
- I built no more than five features.
- I wrote one line for why each feature is known at decision time.
- I showed leakage and then removed it.
- I wrote one clear limitation.
